In [2]:
X = []
y = []
with open('Iris.csv', 'r') as file:
    reader = csv.reader(file)
    header = next(reader)
    print(header)
    target_feature = 5
    index = 0
    for row in reader:
        data = []
        for i in range(len(row)):
            if i != index and i != target_feature:
                data.append(float(row[i]))
            if i == target_feature:
                y.append(row[i])
        X.append(data)
print(len(X))
print(len(X[0]))
print(y[:5])

['Id', 'SepalLengthCm', 'SepalWidthCm', 'PetalLengthCm', 'PetalWidthCm', 'Species']
150
4
['Iris-setosa', 'Iris-setosa', 'Iris-setosa', 'Iris-setosa', 'Iris-setosa']


In [3]:
import random

def stratified_split(X, y, test_ratio=0.2):
    random.seed(42)

    # Step 1: Group record indices by class label
    class_indices = {}

    for i in range(len(y)):
        label = y[i]

        if label not in class_indices:
            class_indices[label] = []

        class_indices[label].append(i)

    train_indices = []
    test_indices = []

    # Step 2: Split each class separately
    for label in class_indices:
        indices = class_indices[label]

        random.shuffle(indices)

        test_size = round(len(indices) * test_ratio)

        test_indices.extend(indices[:test_size])
        train_indices.extend(indices[test_size:])

    #Shuffle the final train and test indices
    
    random.shuffle(train_indices)
    random.shuffle(test_indices)

    # Step 3: Build the datasets using the same indices
    X_train = []
    y_train = []
    X_test = []
    y_test = []

    for i in train_indices:
        X_train.append(X[i])
        y_train.append(y[i])

    for i in test_indices:
        X_test.append(X[i])
        y_test.append(y[i])

    

    return X_train, X_test, y_train, y_test

In [4]:
X_train, X_test, y_train, y_test = stratified_split(X, y)

In [5]:
print("Training labels:", y_train)
print("Test labels:", y_test)

Training labels: ['Iris-versicolor', 'Iris-setosa', 'Iris-versicolor', 'Iris-versicolor', 'Iris-setosa', 'Iris-virginica', 'Iris-virginica', 'Iris-setosa', 'Iris-virginica', 'Iris-versicolor', 'Iris-virginica', 'Iris-versicolor', 'Iris-setosa', 'Iris-virginica', 'Iris-versicolor', 'Iris-virginica', 'Iris-virginica', 'Iris-setosa', 'Iris-versicolor', 'Iris-setosa', 'Iris-virginica', 'Iris-versicolor', 'Iris-virginica', 'Iris-setosa', 'Iris-virginica', 'Iris-setosa', 'Iris-versicolor', 'Iris-versicolor', 'Iris-setosa', 'Iris-versicolor', 'Iris-setosa', 'Iris-setosa', 'Iris-setosa', 'Iris-versicolor', 'Iris-virginica', 'Iris-virginica', 'Iris-setosa', 'Iris-virginica', 'Iris-setosa', 'Iris-setosa', 'Iris-versicolor', 'Iris-setosa', 'Iris-versicolor', 'Iris-virginica', 'Iris-virginica', 'Iris-versicolor', 'Iris-versicolor', 'Iris-setosa', 'Iris-versicolor', 'Iris-versicolor', 'Iris-versicolor', 'Iris-virginica', 'Iris-versicolor', 'Iris-virginica', 'Iris-virginica', 'Iris-setosa', 'Iris-se

In [7]:
labels = set(y_train)
for label in labels:
    print(label, ":",  y_train.count(label) / len(y_train))

Iris-setosa : 0.3333333333333333
Iris-versicolor : 0.3333333333333333
Iris-virginica : 0.3333333333333333


In [8]:

for label in labels:
    print(label, ":",  y_test.count(label) / len(y_test))

Iris-setosa : 0.3333333333333333
Iris-versicolor : 0.3333333333333333
Iris-virginica : 0.3333333333333333


In [11]:
def calculate_priors(y_train):
    priors = {}
    labels = set(y_train)
    for label in labels:
        priors[label] = y_train.count(label) / len(y_train)
    return priors

In [12]:
    
priors = calculate_priors(y_train)
print(priors)

{'Iris-setosa': 0.3333333333333333, 'Iris-versicolor': 0.3333333333333333, 'Iris-virginica': 0.3333333333333333}


In [13]:
def calculate_mean(values):
    total = 0
    for value in values:
        total += value
    return total / len(values)

In [14]:
def calculate_std(values, mean):
    squared_sum = 0
    for value in values:
        diff = value - mean
        squared_sum += diff * diff
    return (squared_sum / len(values)) ** 0.5


In [17]:
def fit_scalar(X):
    means = []
    stds = []

    for j in range(len(X[0])):
        col = []
        for i in range(len(X)):
            col.append(X[i][j])
        mean_col = calculate_mean(col)
        means.append(mean_col)
        stds.append(calculate_std(col, mean_col))

    return means, stds



In [18]:
def cal_stats(X_train, y_train):
    statistics = {}
    labels = set(y_train)

    for label in labels:
        class_samples = []

        # Collect samples belonging to this class
        for i in range(len(y_train)):
            if y_train[i] == label:
                class_samples.append(X_train[i])

        # Reuse your existing function
        means, stds = fit_scalar(class_samples)

        statistics[label] = (means, stds)

    return statistics

In [109]:
statistics = cal_stats(X_train, y_train)
print(statistics)

for label, (means, stds) in statistics.items():
    print("\nClass:", label)

    for j in range(len(means)):
        print(
            "Feature", j + 1,
            "Mean:", round(means[j], 3),
            "Std:", round(stds[j], 3)
        )

{'Iris-setosa': (array([4.985 , 3.4025, 1.48  , 0.2525]), array([0.30458989, 0.39590876, 0.1584298 , 0.1161626 ])), 'Iris-virginica': (array([6.61, 2.98, 5.58, 2.04]), array([0.67631354, 0.3494281 , 0.56885851, 0.26532998])), 'Iris-versicolor': (array([5.93  , 2.75  , 4.2525, 1.32  ]), array([0.47074409, 0.30495901, 0.43759999, 0.18466185]))}

Class: Iris-setosa
Feature 1 Mean: 4.985 Std: 0.305
Feature 2 Mean: 3.402 Std: 0.396
Feature 3 Mean: 1.48 Std: 0.158
Feature 4 Mean: 0.252 Std: 0.116

Class: Iris-virginica
Feature 1 Mean: 6.61 Std: 0.676
Feature 2 Mean: 2.98 Std: 0.349
Feature 3 Mean: 5.58 Std: 0.569
Feature 4 Mean: 2.04 Std: 0.265

Class: Iris-versicolor
Feature 1 Mean: 5.93 Std: 0.471
Feature 2 Mean: 2.75 Std: 0.305
Feature 3 Mean: 4.252 Std: 0.438
Feature 4 Mean: 1.32 Std: 0.185


In [20]:
import math

def gaussian_probability(x, mean, std):
    if std == 0:
        std = 1e-9

    exponent = math.exp(
        -((x - mean) ** 2) / (2 * std ** 2)
    )

    probability = (
        1 / (std * math.sqrt(2 * math.pi))
    ) * exponent

    return probability

In [21]:
def predict_one(sample, priors, statistics):
    scores = {}

    for label in priors:
        # Start with log prior probability
        score = math.log(priors[label])

        means, stds = statistics[label]

        # Add log likelihood of each feature
        for j in range(len(sample)):
            probability = gaussian_probability(
                sample[j],
                means[j],
                stds[j]
            )

            score += math.log(max(probability, 1e-300))

        scores[label] = score

    # Return the class with the highest score
    best_label = max(scores, key=scores.get)

    return best_label

In [22]:
sample = X_test[0]

prediction = predict_one(sample, priors, statistics)

print("Flower measurements:", sample)
print("Actual species:", y_test[0])
print("Predicted species:", prediction)

Flower measurements: [4.8, 3.4, 1.6, 0.2]
Actual species: Iris-setosa
Predicted species: Iris-setosa


In [23]:
def predict(X_test, priors, statistics):
    predictions = []

    for sample in X_test:
        result = predict_one(sample, priors, statistics)
        predictions.append(result)

    return predictions

In [24]:
def calculate_accuracy(y_test, predictions):
    correct = 0

    for i in range(len(y_test)):
        if y_test[i] == predictions[i]:
            correct += 1

    accuracy = correct / len(y_test) * 100

    return accuracy

In [25]:
predictions = predict(X_test, priors, statistics)

accuracy = calculate_accuracy(y_test, predictions)

print("Actual labels:    ", y_test)
print("Predicted labels: ", predictions)
print("Accuracy:", accuracy, "%")

Actual labels:     ['Iris-setosa', 'Iris-setosa', 'Iris-virginica', 'Iris-versicolor', 'Iris-virginica', 'Iris-setosa', 'Iris-versicolor', 'Iris-versicolor', 'Iris-virginica', 'Iris-versicolor', 'Iris-virginica', 'Iris-setosa', 'Iris-versicolor', 'Iris-virginica', 'Iris-versicolor', 'Iris-virginica', 'Iris-virginica', 'Iris-versicolor', 'Iris-setosa', 'Iris-virginica', 'Iris-versicolor', 'Iris-versicolor', 'Iris-setosa', 'Iris-setosa', 'Iris-setosa', 'Iris-setosa', 'Iris-virginica', 'Iris-versicolor', 'Iris-virginica', 'Iris-setosa']
Predicted labels:  ['Iris-setosa', 'Iris-setosa', 'Iris-virginica', 'Iris-versicolor', 'Iris-virginica', 'Iris-setosa', 'Iris-versicolor', 'Iris-versicolor', 'Iris-virginica', 'Iris-versicolor', 'Iris-virginica', 'Iris-setosa', 'Iris-versicolor', 'Iris-versicolor', 'Iris-versicolor', 'Iris-virginica', 'Iris-virginica', 'Iris-versicolor', 'Iris-setosa', 'Iris-virginica', 'Iris-versicolor', 'Iris-versicolor', 'Iris-setosa', 'Iris-setosa', 'Iris-setosa', 'Iri

In [26]:
for i in range(len(y_test)):
    if y_test[i] != predictions[i]:
        print("Sample:", X_test[i])
        print("Actual:", y_test[i])
        print("Predicted:", predictions[i])
        print()

Sample: [6.0, 2.2, 5.0, 1.5]
Actual: Iris-virginica
Predicted: Iris-versicolor

Sample: [4.9, 2.5, 4.5, 1.7]
Actual: Iris-virginica
Predicted: Iris-versicolor



In [27]:
def calculate_precision_recall(y_test, predictions, classes):
    for label in classes:
        tp = 0
        fp = 0
        fn = 0

        for i in range(len(y_test)):
            if y_test[i] == label and predictions[i] == label:
                tp += 1
            elif y_test[i] != label and predictions[i] == label:
                fp += 1
            elif y_test[i] == label and predictions[i] != label:
                fn += 1

        if tp + fp == 0:
            precision = 0
        else:
            precision = tp / (tp + fp)

        if tp + fn == 0:
            recall = 0
        else:
            recall = tp / (tp + fn)

        if precision + recall == 0:
            f1 = 0
        else:
            f1 = 2 * precision * recall / (precision + recall)

        print("\nClass:", label)
        print("TP:", tp, "FP:", fp, "FN:", fn)
        print("Precision:", round(precision, 3))
        print("Recall:", round(recall, 3))
        print("F1-score:", round(f1, 3))

In [28]:
def confusion_matrix(y_test, predictions, classes):
    matrix = []

    for i in range(len(classes)):
        row = [0] * len(classes)
        matrix.append(row)

    for i in range(len(y_test)):
        actual = y_test[i]
        predicted = predictions[i]

        actual_index = classes.index(actual)
        predicted_index = classes.index(predicted)

        matrix[actual_index][predicted_index] += 1

    return matrix

In [29]:
classes = list(set(y_train))

matrix = confusion_matrix(y_test, predictions, classes)

print("Classes:", classes)
for row in matrix:
    print(row)

Classes: ['Iris-setosa', 'Iris-versicolor', 'Iris-virginica']
[10, 0, 0]
[0, 10, 0]
[0, 2, 8]


In [30]:
def calculate_precision_recall(y_test, predictions, classes):
    for label in classes:
        tp = 0
        fp = 0
        fn = 0

        for i in range(len(y_test)):
            if y_test[i] == label and predictions[i] == label:
                tp += 1
            elif y_test[i] != label and predictions[i] == label:
                fp += 1
            elif y_test[i] == label and predictions[i] != label:
                fn += 1

        if tp + fp == 0:
            precision = 0
        else:
            precision = tp / (tp + fp)

        if tp + fn == 0:
            recall = 0
        else:
            recall = tp / (tp + fn)

        if precision + recall == 0:
            f1 = 0
        else:
            f1 = 2 * precision * recall / (precision + recall)

        print("\nClass:", label)
        print("TP:", tp, "FP:", fp, "FN:", fn)
        print("Precision:", round(precision, 3))
        print("Recall:", round(recall, 3))
        print("F1-score:", round(f1, 3))

In [31]:
calculate_precision_recall(y_test, predictions, classes)


Class: Iris-setosa
TP: 10 FP: 0 FN: 0
Precision: 1.0
Recall: 1.0
F1-score: 1.0

Class: Iris-versicolor
TP: 10 FP: 2 FN: 0
Precision: 0.833
Recall: 1.0
F1-score: 0.909

Class: Iris-virginica
TP: 8 FP: 0 FN: 2
Precision: 1.0
Recall: 0.8
F1-score: 0.889
